# A2AJ Canadian Hansard — Hugging Face Dataset

> **Alpha — work in progress.** This is an alpha version of a work in progress academic research project. It is likely to undergo significant revisions.

**Documentation:**

- Canadian Hansard Dataset: https://huggingface.co/datasets/a2aj/hansard
- Hugging Face Datasets: https://huggingface.co/docs/datasets/main/en/index

Hugging Face Datasets is a library that provides easy access to datasets stored in a standardized format and ready to use with just a few lines of code. Instead of downloading files manually and dealing with different formats, the Datasets library handles the heavy lifting — loading data efficiently, managing memory, and providing tools for processing.

The A2AJ (Access to Algorithmic Justice) Canadian Hansard dataset provides bulk access to parliamentary Hansard transcripts — the official record of debate — for Canadian legislatures. The text is parsed into structured rows with **one row per intervention** (a speaker turn or a procedural note), alongside metadata such as the sitting date, speaker, order/subject of business, and jurisdiction. It currently covers the Legislative Assembly of Ontario.

The dataset is built for **computational legal and political research**: programmatically searching the parliamentary record for patterns, tracking how debate and policy evolve over time and in light of case law, assembling evidence of legislative purpose and intent, and linking parliamentary discussion to legislation and case law.

This notebook shows how to load a jurisdiction, convert it to a pandas DataFrame, explore the structure, and run a few example research queries.

**Note:** This bulk approach is ideal for analysis, machine learning, or offline work. If you only need a few specific documents, you can also read the Parquet file directly (shown below).


## Setup

In [ ]:
# install required packages if not already installed
# !pip install datasets pandas

from datasets import load_dataset
import pandas as pd

## Load a specific legislature

Each jurisdiction is a separate **config**, each with a single `train` split.

Current options are:

| Config | Legislature |
|---|---|
| `ontario` | Legislative Assembly of Ontario |

In [ ]:
# load one jurisdiction (config), e.g. Ontario
hansard = load_dataset("a2aj/hansard", "ontario", split="train")

# convert to a DataFrame
df = hansard.to_pandas()
df.head(5)

## Alternative: read the Parquet files directly

The dataset is also offered in Parquet, with one file per jurisdiction. You can read a file directly with pandas (no `datasets` dependency) — handy for quick local use.

In [ ]:
import pandas as pd

df = pd.read_parquet("hf://datasets/a2aj/hansard/ontario/train.parquet")
df.head(5)

## Explore the data structure

Each row is one intervention. The fields are:

| Field | Description |
|---|---|
| `ID` | Stable per-row identifier (sitting date + sequence number). |
| `Date` | Sitting date (YYYY-MM-DD). |
| `jurisdiction` | Which legislature (e.g. `ontario`). |
| `chamber` | Chamber within the legislature (e.g. `legislative_assembly`). |
| `language` | Language of the source page (e.g. `en`). |
| `OrderofBusiness` | High-level section heading (the order of business). |
| `SubjectofBusiness` | Specific topic / matter within the order of business. |
| `PersonSpeaking` | Speaker label, or empty for procedural rows. |
| `intervention_type` | `speech` (a speaker turn) or `procedural` (a procedural note). |
| `Intervention` | The text of the intervention. |
| `upstream_license` | License / attribution terms from the source legislature. |
| `source_url` | The day-page URL this row was parsed from. |

In [ ]:
print("rows:", len(df))
print("columns:", list(df.columns))
print("date range:", df["Date"].min(), "->", df["Date"].max())

# speech vs procedural
df["intervention_type"].value_counts()

In [ ]:
# most frequent speakers
df["PersonSpeaking"].value_counts().head(10)

## Example research queries

A few illustrative patterns. Because every intervention is a row with a speaker, date, topic, and full text, you can slice the parliamentary record however your research question requires.

In [ ]:
# 1) Keyword search across the full text of interventions
term = "climate change"
hits = df[df["Intervention"].str.contains(term, case=False, na=False)]
print(f"{len(hits):,} interventions mention '{term}'")
hits[["Date", "PersonSpeaking", "OrderofBusiness", "Intervention"]].head(5)

In [ ]:
# 2) Filter to a date range (e.g. 2024 onward)
dates = pd.to_datetime(df["Date"], errors="coerce")
recent = df[dates >= "2024-01-01"]
print(f"{len(recent):,} interventions since 2024-01-01")
recent.head(5)

In [ ]:
# 3) How a topic is discussed over time (mentions per year)
term = "housing"
mask = df["Intervention"].str.contains(term, case=False, na=False)
years = pd.to_datetime(df.loc[mask, "Date"], errors="coerce").dt.year
years.value_counts().sort_index()

In [ ]:
# 4) Everything a particular speaker said on a given day
day = df[(df["Date"] == "2024-03-21") & (df["intervention_type"] == "speech")]
day[["PersonSpeaking", "SubjectofBusiness", "Intervention"]].head(10)

## Licensing

The code used to create the dataset is under an open-source MIT license. Users must also comply with the upstream licenses recorded in the `upstream_license` field, which may include limits on commercial use and other restrictions. The texts here are unofficial copies — for authoritative versions, consult the `source_url`. See the [dataset card](https://huggingface.co/datasets/a2aj/hansard) for full details.

Contact: a2aj@yorku.ca